# 01 Data Cleaning and Summary Generation

I use this notebook to show how I move from the raw ACLED export to the cleaned event-level dataset and the three summary CSV files used in the report. I keep the cleaning logic here so the reader can see exactly how the variables in the analysis are created.

Outputs generated here:

- `data/acled_mena_processed.csv`
- `data/yearly_shift_summary.csv`
- `data/spatial_bucket_summary.csv`
- `data/country_pattern_summary.csv`


## Step 0: Set up file paths and packages

I set up the project folder, package imports, and file paths so the notebook can run from either the local project folder or a freshly cloned GitHub copy. I also check that the raw CSV is a real data file rather than a Git LFS pointer, because the cleaning steps cannot run if the large data file has not been downloaded.


In [ ]:
from functools import lru_cache
from pathlib import Path
from typing import Dict
import json
import re
import subprocess
import sys

import pandas as pd

REPO_URL = "https://github.com/lexieliujy/POLI3148_PS1.git"


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "docs" / "index.html").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root folder.")


try:
    ROOT_DIR = find_project_root(Path.cwd())
except FileNotFoundError:
    ROOT_DIR = Path.cwd() / "POLI3148_PS1"
    if not (ROOT_DIR / "data").exists():
        subprocess.run(["git", "clone", REPO_URL, str(ROOT_DIR)], check=True)

CODE_DIR = ROOT_DIR / "code"
DATA_DIR = ROOT_DIR / "data"
VENDOR_DIR = CODE_DIR / "vendor"
for path in [CODE_DIR, VENDOR_DIR]:
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

import geopandas as gpd

RAW_DATA_PATH = DATA_DIR / "ACLED Data_MENA_Raw.csv"
PROCESSED_PATH = DATA_DIR / "acled_mena_processed.csv"
YEARLY_SUMMARY_PATH = DATA_DIR / "yearly_shift_summary.csv"
SPATIAL_SUMMARY_PATH = DATA_DIR / "spatial_bucket_summary.csv"
COUNTRY_SUMMARY_PATH = DATA_DIR / "country_pattern_summary.csv"
BOUNDARY_DATA_PATH = DATA_DIR / "support" / "ne_50m_admin_0_countries.zip"

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f"Raw data file not found: {RAW_DATA_PATH}")
if RAW_DATA_PATH.read_text(errors="ignore", encoding="utf-8")[:80].startswith("version https://git-lfs"):
    raise RuntimeError(
        "The raw CSV is still a Git LFS pointer. Run `git lfs pull` in the project folder, "
        "or download the LFS files from GitHub before running the cleaning notebook."
    )

RAW_DATA_PATH


## Step 1: Define the cleaning rules

I define the main cleaning rules before loading the data so the assumptions are visible. I keep only ACLED events in `Middle East` and `Northern Africa`, use a 50-kilometer threshold for border proximity, list the capital city for each country, identify the columns required for the analysis, and create a text pattern for border-related wording in event notes.


In [ ]:
TARGET_REGIONS = ["Middle East", "Northern Africa"]
BORDER_DISTANCE_KM = 50

CAPITALS_BY_COUNTRY: Dict[str, str] = {
    "Algeria": "Algiers",
    "Bahrain": "Manama",
    "Egypt": "Cairo",
    "Iran": "Tehran",
    "Iraq": "Baghdad",
    "Israel": "Jerusalem",
    "Jordan": "Amman",
    "Kuwait": "Kuwait City",
    "Lebanon": "Beirut",
    "Libya": "Tripoli",
    "Morocco": "Rabat",
    "Oman": "Muscat",
    "Saudi Arabia": "Riyadh",
    "Sudan": "Khartoum",
    "Syria": "Damascus",
    "Turkey": "Ankara",
    "Tunisia": "Tunis",
    "United Arab Emirates": "Abu Dhabi",
    "Yemen": "Sanaa",
}

REQUIRED_COLUMNS = [
    "event_date",
    "year",
    "event_type",
    "sub_event_type",
    "country",
    "admin1",
    "location",
    "latitude",
    "longitude",
    "notes",
    "fatalities",
]

BORDER_PATTERN = re.compile(
    r"\bborder\b|cross-border|cross border|frontier|boundary|border crossing",
    re.IGNORECASE,
)


### 1.1 Define helper functions for raw data and capital areas

I define small helper functions to keep the later cleaning pipeline readable. `load_raw_data()` loads the ACLED CSV, and `is_capital_area()` checks whether an event location is the named capital city or a sub-location inside that capital.


In [ ]:
def load_raw_data() -> pd.DataFrame:
    return pd.read_csv(RAW_DATA_PATH)


def is_capital_area(row: pd.Series) -> bool:
    capital = CAPITALS_BY_COUNTRY.get(row["country"])
    if not capital or pd.isna(row["location"]):
        return False
    location = str(row["location"])
    return location == capital or location.startswith(f"{capital} -")


### 1.2 Measure distance to international borders

I use Natural Earth country boundaries to calculate each event's distance to the nearest international land border. I first load the MENA country geometries, then build border lines by intersecting each country's boundary with neighboring country boundaries. Finally, I convert event coordinates into the same projected coordinate system and calculate distance in kilometers.


In [ ]:
def load_mena_country_geometries() -> gpd.GeoDataFrame:
    world = gpd.read_file(f"zip://{BOUNDARY_DATA_PATH}")[["ADMIN", "geometry"]].copy()
    world = world.rename(columns={"ADMIN": "country_name"})
    mena_countries = list(CAPITALS_BY_COUNTRY.keys()) + ["Palestine"]
    world = world[world["country_name"].isin(mena_countries)].copy()
    world = world.dissolve(by="country_name", as_index=False)
    return world.to_crs(3395)


def build_country_border_lines() -> dict[str, object]:
    countries = load_mena_country_geometries()
    border_lines: dict[str, object] = {}
    for _, row in countries.iterrows():
        country = row["country_name"]
        own_geometry = row.geometry
        neighbors_union = countries.loc[countries["country_name"] != country, "geometry"].union_all()
        border_lines[country] = own_geometry.boundary.intersection(neighbors_union)
    return border_lines


def compute_border_distance_km(df: pd.DataFrame) -> pd.Series:
    border_lines = build_country_border_lines()
    geo_df = gpd.GeoDataFrame(
        df[["country", "longitude", "latitude"]].copy(),
        geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
        crs="EPSG:4326",
    ).to_crs(3395)
    distances = pd.Series(index=df.index, dtype="float64")
    for country in sorted(df["country"].dropna().unique()):
        border_geometry = border_lines.get(country)
        idx = geo_df["country"].eq(country)
        if border_geometry is None or border_geometry.is_empty:
            distances.loc[idx] = float("inf")
        else:
            distances.loc[idx] = geo_df.loc[idx, "geometry"].distance(border_geometry) / 1000.0
    return distances


### 1.3 Build the processed event-level dataset

I put the full cleaning pipeline into `prepare_analysis_data()` so the processed CSV can be recreated in one step. I load the raw data, filter to the target regions, drop rows missing core fields, parse dates, create indicators for capital areas, remote violence, battles, border proximity, and the report's two main proxy variables.


In [ ]:
def prepare_analysis_data() -> pd.DataFrame:
    df = load_raw_data().copy()
    df = df[df["region"].isin(TARGET_REGIONS)].copy()
    df = df.dropna(subset=REQUIRED_COLUMNS).copy()
    df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")
    df = df.dropna(subset=["event_date"]).copy()
    df["notes"] = df["notes"].astype(str)

    df["capital_city"] = df["country"].map(CAPITALS_BY_COUNTRY)
    df["is_capital_area"] = df.apply(is_capital_area, axis=1)
    df["is_border_mentioned"] = df["notes"].str.contains(BORDER_PATTERN, regex=True)
    df["is_remote"] = df["event_type"].eq("Explosions/Remote violence")
    df["is_battle"] = df["event_type"].eq("Battles")

    df["border_distance_km"] = compute_border_distance_km(df)
    df["is_border_proximate"] = df["border_distance_km"].le(BORDER_DISTANCE_KM)

    df["capital_remote_event"] = df["is_capital_area"] & df["is_remote"]
    df["border_battle_event"] = df["is_border_proximate"] & df["is_battle"]
    df["capital_targeting_proxy"] = df["capital_remote_event"].map({True: "Yes", False: "No"})
    df["border_clash_proxy"] = df["border_battle_event"].map({True: "Yes", False: "No"})

    df["spatial_bucket"] = "Other areas"
    df.loc[df["is_border_proximate"], "spatial_bucket"] = "Border-proximate areas"
    df.loc[df["is_capital_area"], "spatial_bucket"] = "Capital areas"
    return df


### 1.4 Save the processed event-level CSV

I run the cleaning function and write the cleaned event-level dataset to `acled_mena_processed.csv`. This file preserves the row-level event data while adding the variables needed for the analysis.


In [ ]:
processed = prepare_analysis_data()
processed.to_csv(PROCESSED_PATH, index=False)

processed.shape, PROCESSED_PATH


## Step 2: Create the report sample

I create the report sample by keeping observations from 1997 through 2024. I do this because the final report excludes 2025 records and discusses a completed 1997-2024 period. The processed file can still contain the broader cleaned dataset, but the summary tables and report figures use this narrower sample.


In [ ]:
report_df = processed[processed["year"].between(1997, 2024)].copy()

{
    "processed_rows": len(processed),
    "report_rows_1997_2024": len(report_df),
    "min_year": int(report_df["year"].min()),
    "max_year": int(report_df["year"].max()),
}


## Step 3: Build and write summary CSV files

I create three summary files from `report_df` so the report can use smaller and easier-to-read tables. These files summarize the same event-level data by year, spatial bucket, and country.


### 3.1 Yearly summary

I group events by `year` to build the time-series table used in the report. I count all events, battles, remote violence events, air/drone strikes, capital-area remote attacks, and border-proximate battles. I then calculate percentage columns so the figures can compare shares rather than only raw counts.


In [ ]:
years = pd.Index(
    range(int(report_df["year"].min()), int(report_df["year"].max()) + 1),
    name="year",
)
yearly = pd.DataFrame(index=years)

yearly["all_events"] = report_df.groupby("year").size()
yearly["all_battles"] = report_df[report_df["is_battle"]].groupby("year").size()
yearly["all_remote_events"] = report_df[report_df["is_remote"]].groupby("year").size()
yearly["drone_events"] = report_df[
    report_df["sub_event_type"].eq("Air/drone strike")
].groupby("year").size()
yearly["capital_remote_events"] = report_df[
    report_df["capital_remote_event"]
].groupby("year").size()
yearly["border_battle_events"] = report_df[
    report_df["border_battle_event"]
].groupby("year").size()
yearly["capital_remote_fatalities"] = report_df[
    report_df["capital_remote_event"]
].groupby("year")["fatalities"].sum()
yearly["border_battle_fatalities"] = report_df[
    report_df["border_battle_event"]
].groupby("year")["fatalities"].sum()

yearly = yearly.fillna(0).astype(int).reset_index()
yearly["capital_share_of_remote_pct"] = (
    yearly["capital_remote_events"] / yearly["all_remote_events"].replace(0, pd.NA) * 100
).fillna(0).round(2)
yearly["border_share_of_battles_pct"] = (
    yearly["border_battle_events"] / yearly["all_battles"].replace(0, pd.NA) * 100
).fillna(0).round(2)
yearly["drone_share_of_remote_pct"] = (
    yearly["drone_events"] / yearly["all_remote_events"].replace(0, pd.NA) * 100
).fillna(0).round(2)

yearly.to_csv(YEARLY_SUMMARY_PATH, index=False)
yearly.head()


### 3.2 Spatial-bucket summary

I group events by `spatial_bucket` to compare capital areas, border-proximate areas, and other areas. For each bucket, I count total events, fatalities, remote events, and battles. I then calculate the share of events in that bucket that are remote violence or battles.


#### Border-proximate classification

I calculate `is_border_proximate` by measuring each event coordinate's distance to the nearest international land border using Natural Earth country boundaries. I classify an event as border-proximate when `border_distance_km <= 50`. I use this threshold to capture events likely shaped by frontier or cross-border dynamics without relying only on whether the event note mentions the word "border." I then create `spatial_bucket`: capital-area events are labeled `Capital areas`, non-capital border-proximate events are labeled `Border-proximate areas`, and all remaining events are labeled `Other areas`.


In [ ]:
spatial = (
    report_df.groupby("spatial_bucket")
    .agg(
        events=("event_id_cnty", "count"),
        fatalities=("fatalities", "sum"),
        remote_events=("is_remote", "sum"),
        battles=("is_battle", "sum"),
    )
    .reset_index()
)
spatial["remote_share_pct"] = (
    spatial["remote_events"] / spatial["events"] * 100
).round(2)
spatial["battle_share_pct"] = (
    spatial["battles"] / spatial["events"] * 100
).round(2)
spatial = spatial.sort_values("events", ascending=False)

spatial.to_csv(SPATIAL_SUMMARY_PATH, index=False)
spatial


### 3.3 Country-level summary

I group events by `country` to show where the main patterns are concentrated. For each country, I calculate total events, remote events, capital-area remote attacks, border-proximate battles, fatalities, and two percentage measures: remote events as a share of all events, and capital-area remote attacks as a share of remote events.


In [ ]:
country = (
    report_df.groupby("country")
    .agg(
        total_events=("event_id_cnty", "count"),
        remote_events=("is_remote", "sum"),
        capital_remote_events=("capital_remote_event", "sum"),
        border_battle_events=("border_battle_event", "sum"),
        fatalities=("fatalities", "sum"),
    )
    .reset_index()
)
country["remote_share_pct"] = (
    country["remote_events"] / country["total_events"] * 100
).round(2)
country["capital_remote_share_pct"] = (
    country["capital_remote_events"] / country["remote_events"].replace(0, pd.NA) * 100
).fillna(0).round(2)
country = country.sort_values("total_events", ascending=False)

country.to_csv(COUNTRY_SUMMARY_PATH, index=False)
country.head()


### 3.4 Output paths

I display the output paths to confirm where the three summary CSV files were written. This is a simple final check that the notebook produced the files used later in the analysis notebook and the HTML report.


In [ ]:
[YEARLY_SUMMARY_PATH, SPATIAL_SUMMARY_PATH, COUNTRY_SUMMARY_PATH]


## Step 4: Sanity checks

I calculate a few headline counts to check that the cleaned sample matches the report narrative. I count all events, border-proximate events, capital-area events, overlap between the two spatial categories, non-capital border-proximate events, and capital-area remote events.


In [ ]:
checks = {
    "all_events_1997_2024": len(report_df),
    "border_proximate_events": int(report_df["is_border_proximate"].sum()),
    "capital_area_events": int(report_df["is_capital_area"].sum()),
    "overlap_events": int((report_df["is_border_proximate"] & report_df["is_capital_area"]).sum()),
    "noncapital_border_proximate_events": int((report_df["is_border_proximate"] & ~report_df["is_capital_area"]).sum()),
    "capital_remote_events": int(report_df["capital_remote_event"].sum()),
}

checks


In [ ]:
yearly.tail()
